# 03 -- Latent-guided sampler

**No fine-tuning of anything.** The DDPM and the Xception encoder are both frozen and loaded from their published checkpoints, unchanged.

Step A trains a small `theta -> z*` head (`LatentPredictor`) that maps the 8 physical conditioning parameters to a target encoder latent. This is NOT a fine-tune of either big model -- it is a lightweight auxiliary network trained from scratch in this notebook, on latents precomputed by the frozen encoder.

Step B adds classifier-guidance-style gradient steps inside sampling: at each low-noise timestep, the predicted `x0` is nudged towards higher cosine similarity with the target latent `z*`.

Step C sweeps the guidance scale and reports physical-metric R^2, SSIM, latent cosine and cycle R^2/MAE per scale -- including the expected failure mode at high guidance scale (texture artefacts, rising latent cosine).

**Kaggle input datasets required (attach all four):**
- `carloscanamejoy/dataset-spines-united-v2` -> `dataset_unificado_v2.npz`
- `carloscanamejoy/weights-xception-model` -> `xception_regressor_torch.pt`
- `carloscanamejoy/weights-models` -> `ddpm_spines_final_39crop.pt`
- `carloscanamejoy/physicalmetrics` -> `metrics.py`

Outputs are written to `/kaggle/working/latent_guided_sampler/`.

## 1. Environment setup

In [ ]:
# Check GPU availability
import os
from pathlib import Path

if Path('/kaggle').exists():
    print('Running on Kaggle')
else:
    print('WARNING: this does not look like Kaggle; continuing anyway')

try:
    import subprocess
    subprocess.run(['nvidia-smi'], check=False)
except Exception as e:
    print(f'Could not run nvidia-smi: {e}')


In [ ]:
# Dependencies. Kaggle usually ships torch/sklearn/matplotlib.
# pytorch-msssim may be missing; install it with a skimage SSIM fallback if the
# install fails (e.g. no internet on the Kaggle session).
try:
    import pytorch_msssim  # noqa: F401
    print('pytorch-msssim available')
except Exception:
    print('pytorch-msssim not installed; attempting install...')
    try:
        import sys, subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pytorch-msssim'])
        print('pytorch-msssim installed')
    except Exception as e:
        print(f'WARNING: could not install pytorch-msssim ({e}). Falling back to skimage SSIM.')

In [ ]:
# On Kaggle, datasets are mounted under /kaggle/input -- no kaggle.json needed.
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
if not KAGGLE_INPUT.exists():
    raise RuntimeError('/kaggle/input does not exist. Attach the datasets from "Add Input" on Kaggle.')

print('Mounted datasets:')
for d in sorted([x for x in KAGGLE_INPUT.iterdir() if x.is_dir()]):
    print(' -', d.name)


In [ ]:
# Resolve every required input file by name -- never hardcode a Kaggle path,
# dataset version suffixes change the mount directory name.
import glob


def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits:
        raise FileNotFoundError(
            f'{name} not found under /kaggle/input. '
            f'Attach the dataset that ships it (see the header markdown cell).'
        )
    return hits[0]


DATASET_PATH  = find_file('dataset_unificado_v2.npz')
METRICS_PATH  = find_file('metrics.py')
ENCODER_CKPT  = find_file('xception_regressor_torch.pt')
DDPM_CKPT     = find_file('ddpm_spines_final_39crop.pt')

print(f'DATASET_PATH : {DATASET_PATH}')
print(f'METRICS_PATH : {METRICS_PATH}')
print(f'ENCODER_CKPT : {ENCODER_CKPT}')
print(f'DDPM_CKPT    : {DDPM_CKPT}')


## 2. Imports and global configuration

In [ ]:
import os
import gc
import sys
import json
import time
import math
import pickle
import random
import importlib.util
import warnings

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

try:
    from pytorch_msssim import ssim as ssim_fn
    SSIM_BACKEND = 'pytorch_msssim'
except Exception:
    from skimage.metrics import structural_similarity as skimage_ssim
    SSIM_BACKEND = 'skimage_fallback'

    def ssim_fn(x, y, data_range=1.0, size_average=True):
        x_np = x.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()
        vals = [skimage_ssim(a[0], b[0], data_range=data_range) for a, b in zip(x_np, y_np)]
        val = float(np.mean(vals)) if size_average else np.asarray(vals, dtype=np.float32)
        return torch.as_tensor(val)

warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device : {DEVICE}')
print(f'SSIM   : {SSIM_BACKEND}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


In [ ]:
# Load metrics.py by path and register it as sys.modules['metrics'] -- the same
# idiom used by the rest of the repo's notebooks.
spec = importlib.util.spec_from_file_location('metrics', METRICS_PATH)
metrics = importlib.util.module_from_spec(spec)
spec.loader.exec_module(metrics)
sys.modules['metrics'] = metrics

# Only the canonical three physical metrics exist in metrics.py; the retired
# susceptibility/specific-heat/exchange-energy metrics were removed upstream and are
# never imported here.
from metrics import (
    MASK, IMG_SIZE as PHYS_SIZE, DDPM_SIZE as PHYS_CANVAS_SIZE,
    topleft_crop, magnetization, spin_correlation, peak_wave_vector,
    physical_metrics_batch, PHYSICAL_METRIC_NAMES, PHYSICAL_METRIC_LABELS,
    masked_mse, masked_ssim, get_structure_label, apply_figure_style, save_figure,
    PARAM_NAMES, PARAM_INDEX,
)

apply_figure_style()
print(f'metrics module loaded from {METRICS_PATH}')
print(f'MASK: {MASK.shape}, disk pixels = {int(MASK.sum())}')
print(f'Physical metrics (canonical set of three): {PHYSICAL_METRIC_NAMES}')


In [ ]:
BEST_HPARAMS = {
    'lr':            2.2640535194211016e-04,
    'batch_size':    128,
    'base_ch':       80,
    'cond_emb_dim':  128,
    'dropout':       0.1,
    'beta_schedule': 'cosine',
    'ema_decay':     0.999,
    'weight_decay':  4.279388675327132e-05,
    'min_snr_gamma': 5.0,
}

WARMUP_EPOCHS  = 3
GRAD_CLIP_NORM = 0.5

IMG_SIZE  = 40
CROP_TO   = 39
COND_DIM  = 8
T_STEPS    = 1000
BETA_START = 1e-4
BETA_END   = 0.02

# --- Step A: LatentPredictor training -------------------------------------
PREDICTOR_EPOCHS = 30
PREDICTOR_LR = 1e-3
PREDICTOR_MSE_WEIGHT = 0.1   # loss = (1 - cos(pred, z_target)) + PREDICTOR_MSE_WEIGHT * MSE

# --- Step B: guided sampling ------------------------------------------------
GUIDANCE_T_MAX = 400
# Guide only while t < GUIDANCE_T_MAX -- same rationale as the cosine loss t-window
# in notebooks 01/02 (SPEC 1.1): at high t, x0_pred carries no usable latent signal.

# --- Step C: guidance-scale sweep -------------------------------------------
# Ordered by information value, NOT numerically. At the matched protocol one guided
# scale costs ~4.7 h (measured: 0.84 s per guided batch-step in the previous run), so the
# full six-scale sweep would be ~25 h against a 12 h session limit. Scale 0 is unguided and
# cheap, and validates the protocol against the reference; scale 8 shows the maximum
# effect. sweep_results is checkpointed after EVERY scale, so a cut keeps what finished.
GUIDANCE_SCALES = [0, 8, 2]
# 0 is the unguided baseline and MUST reproduce the plain DDPM.
DEFAULT_EVAL_SCALE = 1
# Representative scale used for the canonical Section-3 evaluation block (parity plot,
# masked MSE, cycle R^2/MAE/RMSE, per-phase breakdown) -- the full GUIDANCE_SCALES sweep
# above already covers physical R^2/SSIM/latent-cosine/cycle-R2/MAE for every scale.

# A conditional DDPM draws a REALISATION from p(x | theta); it does not reproduce one
# specific reference image. Scoring a single draw against a single reference therefore
# caps R^2 at a value set by the intrinsic conditional variance, even for a perfect model.
# EVAL_K samples per theta are averaged before scoring, which estimates the conditional
# mean instead -- the same estimator as notebooks/evaluation/
# physical-metrics-comparison-4models.ipynb (K=32, N=1000, 50 steps), so the numbers here
# are directly comparable to the 0.745 / 0.832 / 0.909 it reports for this DDPM.
EVAL_K = 32
# Thetas evaluated in the sweep. Cost scales as N_THETA_EVAL * EVAL_K * SAMPLE_STEPS per
# guidance scale, and each guided step is a full Xception forward AND backward at 224x224.
# Matched to notebooks/evaluation physical_metrics_3ddpm_comparison (Kaggle kernel
# carloscanamejoy/physical-metrics-3ddpm-comparison): K=32, N_TARGET=1000 sampled
# PROPORTIONALLY by magnetic structure, 100 DDIM steps. Only with all three aligned are
# the R^2 values comparable to its 0.760 / 0.858 / 0.911 for this same base DDPM.
N_THETA_EVAL = 1000
# The pool the stratified draw selects FROM must be the whole test split, or picking
# N_THETA_EVAL=1000 out of a pre-truncated ~1,018 is not a stratified sample at all. This
# first pass only crops and measures REAL images -- it generates nothing -- so the full
# split is cheap here; the cost is governed by N_THETA_EVAL and GUIDANCE_SCALES.
TEST_FRACTION = 1.0
# Fraction of the internal test split used by evaluation. Lower this (e.g. 0.05) for
# a cheap smoke run.
SAMPLE_STEPS = 100       # matches physical_metrics_3ddpm_comparison (FAST_STEPS=100)

WORK_DIR = '/kaggle/working/latent_guided_sampler'
os.makedirs(WORK_DIR, exist_ok=True)
PREDICTOR_CKPT = f'{WORK_DIR}/latent_predictor.pt'
SWEEP_METRICS_OUT = f'{WORK_DIR}/guidance_sweep_metrics.json'

print('Config loaded:')
print(f'  PREDICTOR_EPOCHS = {PREDICTOR_EPOCHS}   PREDICTOR_LR = {PREDICTOR_LR}')
print(f'  GUIDANCE_T_MAX   = {GUIDANCE_T_MAX}')
print(f'  GUIDANCE_SCALES  = {GUIDANCE_SCALES}')
print(f'  WORK_DIR         = {WORK_DIR}')


## 3. Dataset: load, global normalisation constants, 70/15/15 split

In [ ]:
data   = np.load(DATASET_PATH)
imgs   = data['img'].astype(np.float32)
params = data['params'].astype(np.float32)
labels = np.asarray(data['labels']) if 'labels' in data.files else None
if imgs.ndim == 3:
    imgs = imgs[..., np.newaxis]

N = len(imgs)
print(f'Dataset total: {N:,}')
print(f'  imgs   : {imgs.shape}  dtype={imgs.dtype}  range=[{imgs.min():.3f}, {imgs.max():.3f}]')
print(f'  params : {params.shape}  dtype={params.dtype}')
if labels is not None:
    print(f'  labels : {labels.shape}  unique clusters = {sorted(set(labels.tolist()))}')
else:
    print('  labels : not present in this .npz -- per-phase breakdown will be skipped')

# Global min/max of the RAW physical images -- these are the constants the DDPM
# dataset used for its [-1, 1] normalisation (SPEC 0.4), and are needed again to
# undo that normalisation before feeding a DDPM output to the Xception encoder.
IMG_MIN = float(imgs.min())
IMG_MAX = float(imgs.max())
print(f'  IMG_MIN={IMG_MIN:.4f}  IMG_MAX={IMG_MAX:.4f}')


In [ ]:
def make_split(subsample_frac, seed=SEED):
    """70/15/15 split with a MinMaxScaler fitted on the train fold (DDPM conditioning scaler)."""
    rng = np.random.RandomState(seed)
    sub_idx = rng.choice(N, size=int(N * subsample_frac), replace=False)
    imgs_s, params_s = imgs[sub_idx], params[sub_idx]

    idx_all = np.arange(len(sub_idx))
    idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.30, random_state=seed)
    idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, random_state=seed)

    sc = MinMaxScaler()
    p_tr = sc.fit_transform(params_s[idx_tr]).astype(np.float32)
    p_va = sc.transform(params_s[idx_va]).astype(np.float32)
    p_te = sc.transform(params_s[idx_te]).astype(np.float32)

    return {
        'imgs_tr': imgs_s[idx_tr], 'p_tr': p_tr, 'idx_tr': sub_idx[idx_tr],
        'imgs_va': imgs_s[idx_va], 'p_va': p_va, 'idx_va': sub_idx[idx_va],
        'imgs_te': imgs_s[idx_te], 'p_te': p_te, 'idx_te': sub_idx[idx_te],
        'scaler': sc,
    }


SPLIT = make_split(subsample_frac=1.0, seed=SEED)
print(f"train={len(SPLIT['p_tr']):,}  val={len(SPLIT['p_va']):,}  test={len(SPLIT['p_te']):,}")


## 4. PyTorch dataset (reflect-pad 39x39 -> 40x40, no interpolation)

In [ ]:
class SpinesDataset(Dataset):
    """39x39 image reflect-padded to 40x40: F.pad(x, (0, 1, 0, 1), mode='reflect').
    Padding is applied on the RIGHT and BOTTOM edges only, so
    ``img[..., :39, :39]`` is an exact top-left crop back to the original pixels
    (never a centre crop -- see metrics.topleft_crop).
    """

    def __init__(self, imgs_arr, params_arr, img_size=40):
        imgs_t = torch.from_numpy(imgs_arr).permute(0, 3, 1, 2).float()
        H, W = imgs_t.shape[-2], imgs_t.shape[-1]
        if H != img_size or W != img_size:
            pad_h, pad_w = img_size - H, img_size - W
            assert pad_h >= 0 and pad_w >= 0
            imgs_t = F.pad(imgs_t, (0, pad_w, 0, pad_h), mode='reflect')
        mn, mx = imgs_t.min(), imgs_t.max()
        imgs_t = (imgs_t - mn) / (mx - mn + 1e-8)
        imgs_t = imgs_t * 2.0 - 1.0
        self.imgs = imgs_t.float()
        self.params = torch.from_numpy(params_arr).float()

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        return self.imgs[i], self.params[i]


def make_dataloaders(split_dict, batch_size, num_workers=2):
    ds_tr = SpinesDataset(split_dict['imgs_tr'], split_dict['p_tr'], IMG_SIZE)
    ds_va = SpinesDataset(split_dict['imgs_va'], split_dict['p_va'], IMG_SIZE)
    ds_te = SpinesDataset(split_dict['imgs_te'], split_dict['p_te'], IMG_SIZE)
    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True,
                        num_workers=num_workers, pin_memory=True, drop_last=True)
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False,
                        num_workers=num_workers, pin_memory=True)
    return ds_tr, ds_va, ds_te, dl_tr, dl_va, dl_te


_ds_tr, _, _, _dl_tr, _, _ = make_dataloaders(SPLIT, batch_size=BEST_HPARAMS['batch_size'])
_x, _y = next(iter(_dl_tr))
print(f'Smoke test -- img: {_x.shape} [{_x.min():.2f}, {_x.max():.2f}]  cond: {_y.shape}')
del _ds_tr, _dl_tr, _x, _y


## 5. Noise schedule (cosine, per BEST_HPARAMS)

In [ ]:
class DDPMScheduler:
    """Beta schedule for DDPM. Supports 'linear' and 'cosine'."""

    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, schedule='linear', device='cpu'):
        self.T = T
        self.schedule = schedule
        if schedule == 'linear':
            betas = torch.linspace(beta_start, beta_end, T, device=device)
        elif schedule == 'cosine':
            # Nichol & Dhariwal 2021
            steps = T + 1
            s = 0.008
            x = torch.linspace(0, T, steps, device=device)
            alphas_cumprod = torch.cos(((x / T) + s) / (1 + s) * math.pi * 0.5) ** 2
            alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
            betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
            betas = betas.clamp(max=0.999)
        else:
            raise ValueError(f'Unknown schedule: {schedule}')

        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, dim=0)
        self.sqrt_alphas_cumprod = alphas_cumprod.sqrt()
        self.sqrt_one_minus_alphas_cumprod = (1.0 - alphas_cumprod).sqrt()
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
        self.sqrt_recip_alphas = (1.0 / alphas).sqrt()
        self.betas = betas
        self.alphas = alphas
        self.alphas_cumprod = alphas_cumprod
        self.snr = alphas_cumprod / (1.0 - alphas_cumprod)  # SNR_t, for min-SNR weighting

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_a = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)
        sqrt_1a = self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1)
        return sqrt_a * x0 + sqrt_1a * noise, noise

    def min_snr_weight(self, t, gamma=5.0):
        snr_t = self.snr[t]
        return torch.clamp(snr_t, max=gamma) / snr_t

    @torch.no_grad()
    def p_sample_step(self, model, x_t, t_scalar, cond):
        B = x_t.shape[0]
        t_tensor = torch.full((B,), t_scalar, device=x_t.device, dtype=torch.long)
        eps_pred = model(x_t, t_tensor, cond)
        beta_t = self.betas[t_scalar]
        sqrt_ra = self.sqrt_recip_alphas[t_scalar]
        sqrt_1ma = self.sqrt_one_minus_alphas_cumprod[t_scalar]
        mean = sqrt_ra * (x_t - beta_t / sqrt_1ma * eps_pred)
        if t_scalar > 0:
            z = torch.randn_like(x_t)
            sigma = self.posterior_variance[t_scalar].sqrt()
            return mean + sigma * z
        return mean


_sch = DDPMScheduler(T=T_STEPS, schedule=BEST_HPARAMS['beta_schedule'], device=DEVICE)
print(f"Scheduler '{BEST_HPARAMS['beta_schedule']}' OK, "
      f'alphas_cumprod[0]={_sch.alphas_cumprod[0]:.6f}  alphas_cumprod[T-1]={_sch.alphas_cumprod[-1]:.6f}')
del _sch


## 6. Conditional U-Net

In [ ]:
def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / (half - 1))
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeCondEmbedding(nn.Module):
    def __init__(self, t_dim, cond_in, out_dim):
        super().__init__()
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))
        self.c_mlp = nn.Sequential(nn.Linear(cond_in, out_dim), nn.SiLU(), nn.Linear(out_dim, out_dim))

    def forward(self, t, cond):
        t_emb = sinusoidal_embedding(t, self.t_mlp[0].in_features)
        return self.t_mlp(t_emb) + self.c_mlp(cond)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, emb_dim, groups=8, dropout=0.0):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_ch)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, emb):
        h = F.silu(self.norm1(x))
        h = self.conv1(h)
        h = h + self.emb_proj(F.silu(emb))[:, :, None, None]
        h = F.silu(self.norm2(h))
        h = self.dropout(h)
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttention(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, ch)
        self.qkv = nn.Conv2d(ch, ch * 3, 1)
        self.proj = nn.Conv2d(ch, ch, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        q, k, v = self.qkv(h).chunk(3, dim=1)
        q, k, v = q.reshape(B, C, -1), k.reshape(B, C, -1), v.reshape(B, C, -1)
        attn = torch.softmax(torch.bmm(q.transpose(1, 2), k) / math.sqrt(C), dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).reshape(B, C, H, W)
        return x + self.proj(out)


class ConditionalUNet(nn.Module):
    def __init__(self, img_channels=1, base_ch=64, ch_mults=(1, 2, 4), cond_dim=8, emb_dim=128, dropout=0.0):
        super().__init__()
        t_dim = emb_dim
        chs = [base_ch * m for m in ch_mults]
        self.emb = TimeCondEmbedding(t_dim=t_dim, cond_in=cond_dim, out_dim=emb_dim)
        self.conv_in = nn.Conv2d(img_channels, chs[0], 3, padding=1)

        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        in_ch = chs[0]
        self.skip_channels = []
        for i, out_ch in enumerate(chs):
            self.down_blocks.append(nn.ModuleList([
                ResBlock(in_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.skip_channels.append(out_ch)
            self.down_samples.append(
                nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs) - 1 else nn.Identity()
            )
            in_ch = out_ch

        self.mid_block1 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)
        self.mid_attn = SelfAttention(chs[-1])
        self.mid_block2 = ResBlock(chs[-1], chs[-1], emb_dim, dropout=dropout)

        self.up_blocks = nn.ModuleList()
        self.up_samples = nn.ModuleList()
        for i, out_ch in enumerate(reversed(chs)):
            skip_ch = self.skip_channels[-(i + 1)]
            self.up_blocks.append(nn.ModuleList([
                ResBlock(in_ch + skip_ch, out_ch, emb_dim, dropout=dropout),
                ResBlock(out_ch, out_ch, emb_dim, dropout=dropout),
            ]))
            self.up_samples.append(
                nn.ConvTranspose2d(out_ch, out_ch, 4, stride=2, padding=1) if i < len(chs) - 1 else nn.Identity()
            )
            in_ch = out_ch

        self.norm_out = nn.GroupNorm(8, chs[0])
        self.conv_out = nn.Conv2d(chs[0], img_channels, 1)

    def forward(self, x, t, cond):
        emb = self.emb(t, cond)
        h = self.conv_in(x)
        skips = []
        for (rb1, rb2), ds in zip(self.down_blocks, self.down_samples):
            h = rb1(h, emb); h = rb2(h, emb)
            skips.append(h)
            h = ds(h)
        h = self.mid_block1(h, emb); h = self.mid_attn(h); h = self.mid_block2(h, emb)
        for (rb1, rb2), us, skip in zip(self.up_blocks, self.up_samples, reversed(skips)):
            h = torch.cat([h, skip], dim=1)
            h = rb1(h, emb); h = rb2(h, emb)
            h = us(h)
        h = F.silu(self.norm_out(h))
        return self.conv_out(h)


_m = ConditionalUNet(base_ch=BEST_HPARAMS['base_ch'], emb_dim=BEST_HPARAMS['cond_emb_dim'], dropout=BEST_HPARAMS['dropout']).to(DEVICE)
_n = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f'U-Net: {_n/1e6:.2f}M params')
del _m
torch.cuda.empty_cache()


## 7. EMA and sampling helpers

In [ ]:
class EMA:
    """Simple EMA over model parameters."""

    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.data.detach().clone() for n, p in model.named_parameters() if p.requires_grad}

    @torch.no_grad()
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    @torch.no_grad()
    def store_and_copy_to(self, model):
        self._backup = {n: p.data.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                p.data.copy_(self.shadow[n])

    @torch.no_grad()
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self._backup:
                p.data.copy_(self._backup[n])
        self._backup = {}


@torch.no_grad()
def compute_image_metrics(x_gen, x_real):
    """Plain image-space metrics on the 40x40 canvas (MAE/MSE/SSIM) -- NOT physical metrics."""
    mae = (x_gen - x_real).abs().mean().item()
    mse = ((x_gen - x_real) ** 2).mean().item()
    x_g01 = (x_gen + 1.0) / 2.0
    x_r01 = (x_real + 1.0) / 2.0
    ssim_val = ssim_fn(x_g01, x_r01, data_range=1.0, size_average=True).item()
    return mae, mse, ssim_val


@torch.no_grad()
def fast_sample(model, cond, scheduler, n_steps=100, img_size=40):
    B = cond.shape[0]
    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, scheduler.T, scheduler.T // n_steps))[::-1]
    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        eps_pred = model(x, t_tensor, cond)
        sqrt_a = scheduler.sqrt_alphas_cumprod[t_val]
        sqrt_1a = scheduler.sqrt_one_minus_alphas_cumprod[t_val]
        x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
        if t_val > 0:
            t_prev = max(t_val - scheduler.T // n_steps, 0)
            sqrt_a_prev = scheduler.sqrt_alphas_cumprod[t_prev]
            sqrt_1a_prev = scheduler.sqrt_one_minus_alphas_cumprod[t_prev]
            x = sqrt_a_prev * x0_pred + sqrt_1a_prev * eps_pred
        else:
            x = x0_pred
    return x


print('DDPM core (scheduler, U-Net, EMA, fast_sample) defined.')


## 8. Load the published DDPM checkpoint

In [ ]:
model = ConditionalUNet(
    img_channels=1,
    base_ch=BEST_HPARAMS['base_ch'],
    ch_mults=(1, 2, 4),
    cond_dim=COND_DIM,
    emb_dim=BEST_HPARAMS['cond_emb_dim'],
    dropout=BEST_HPARAMS['dropout'],
).to(DEVICE)

scheduler = DDPMScheduler(T=T_STEPS, beta_start=BETA_START, beta_end=BETA_END,
                           schedule=BEST_HPARAMS['beta_schedule'], device=DEVICE)

_ckpt = torch.load(DDPM_CKPT, map_location=DEVICE, weights_only=False)
_state = _ckpt['model'] if isinstance(_ckpt, dict) and 'model' in _ckpt else _ckpt
# Strict by design. This notebook WARM-STARTS from the published checkpoint: a silent
# partial load would leave the U-Net partly randomly initialised while every log line
# still claimed a warm start. A key or shape mismatch means the checkpoint was not
# written by this architecture, and that must stop the run, not print a counter.
model.load_state_dict(_state, strict=True)
print(f'Loaded DDPM checkpoint (strict=True): {DDPM_CKPT}')
print(f'  {len(_state)} tensors into ConditionalUNet '
      f'({sum(p.numel() for p in model.parameters())/1e6:.2f}M params)')

ema = EMA(model, decay=BEST_HPARAMS['ema_decay'])
if isinstance(_ckpt, dict) and _ckpt.get('ema') is not None:
    _ema_skipped = [k for k in _ckpt['ema'] if k not in ema.shadow]
    if _ema_skipped:
        raise RuntimeError(
            f'EMA shadow keys not present in the model: {_ema_skipped[:8]} '
            f'({len(_ema_skipped)} total). The checkpoint does not match this architecture.')
    for k, v in _ckpt['ema'].items():
        ema.shadow[k].copy_(v.to(DEVICE))
    print('  EMA shadow weights loaded from checkpoint.')
else:
    print('  WARNING: no EMA state in checkpoint; EMA shadow initialised from loaded weights.')


## 9. Xception inverse-model encoder

`xception_regressor_torch.pt` is **not** a generic checkpoint: it is produced by
`notebooks/inverse/xception-keras-to-torch.ipynb`, which ports
`modelo_xception_fulldatabaseV3100.h5` layer by layer into the module defined below and only
saves it once the Keras/PyTorch parity test passes (max abs difference < 1e-3 on real images).

So the builder has to be that exact module. Its state-dict keys are flat
(`conv1`, `bn1`, `b2_sc1`, `mid_sc.*`, `head_fc1`, ...) and they do **not** match a timm
`legacy_xception` backbone, where the weights live under `backbone.*` and `bn1`/`bn2` are the
entry-flow `BatchNorm2d(32)`/`BatchNorm2d(64)` rather than the head's `BatchNorm1d(2048)`/
`BatchNorm1d(256)`. Loading this checkpoint into a timm-based module fails with

```
size mismatch for bn1.weight: copying a param with shape torch.Size([32]) from checkpoint,
the shape in current model is torch.Size([2048]).
```

and with `strict=False` it would be worse than a crash: the name collision is the *only* thing
that errors, every backbone weight would be silently dropped and the encoder would run at random
initialisation. Hence `strict=True` below — a mismatch must be loud.

Two details of the port are load-bearing and are reproduced here:

- **TF `'same'` max-pool padding.** Keras splits the padding asymmetrically; a symmetric
  `MaxPool2d(padding=1)` shifts the feature map. `tf_same_maxpool` pads with `-inf` so the
  artificial cells never win a maximum.
- **BatchNorm epsilon.** Keras defaults to `1e-3`, PyTorch to `1e-5`. The conversion notebook
  copied Keras' value onto each module, but `eps` is a plain Python attribute and is **not**
  stored in the state dict — rebuilding the module here has to set it again, otherwise the
  ported weights quietly drift from the Keras evaluator.

In [ ]:
# --- Xception inverse model: exact PyTorch port of the Keras regressor ------
# Mirrors notebooks/inverse/xception-keras-to-torch.ipynb, which is what wrote
# ENCODER_CKPT. Any change here must be mirrored there or the state dict stops loading.

# Keras BatchNormalization defaults to epsilon=1e-3; PyTorch defaults to 1e-5.
# `eps` is not part of the state dict, so it has to be re-declared at build time.
KERAS_BN_EPS = 1e-3


def _bn2d(c):
    return nn.BatchNorm2d(c, eps=KERAS_BN_EPS)


def _bn1d(c):
    return nn.BatchNorm1d(c, eps=KERAS_BN_EPS)


def tf_same_maxpool(x, k=3, s=2):
    """MaxPooling2D(k, strides=s, padding='same') as TensorFlow computes it.

    TF distributes 'same' padding asymmetrically when the input size is even, which
    PyTorch's symmetric `padding=` cannot express. Padding with -inf reproduces how TF
    ignores the artificial cells inside a maximum.
    """
    ih, iw = x.shape[-2], x.shape[-1]
    oh, ow = math.ceil(ih / s), math.ceil(iw / s)
    ph = max((oh - 1) * s + k - ih, 0)
    pw = max((ow - 1) * s + k - iw, 0)
    if ph or pw:
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2), value=float('-inf'))
    return F.max_pool2d(x, k, s)


class SepConv(nn.Module):
    """SeparableConv2D(out, 3, padding='same', use_bias=False): depthwise then pointwise."""

    def __init__(self, cin, cout):
        super().__init__()
        self.depthwise = nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False)
        self.pointwise = nn.Conv2d(cin, cout, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


class XceptionRegressor(nn.Module):
    """Xception (Chollet 2017) + the 8-output regression head of the inverse model.

    Head graph (SPEC 0.4): GlobalAveragePooling2D -> BatchNorm -> Dropout(0.4)
    -> Dense(256, relu) -> BatchNorm -> Dropout(0.3) -> Dense(8, linear).
    ``forward_features`` returns the 256-d latent AFTER the ReLU of Dense(256) and
    BEFORE the final Dense(8) -- this is ``z`` used everywhere in the cosine loss.
    """

    def __init__(self, num_targets=8):
        super().__init__()
        # --- entry flow ---
        self.conv1 = nn.Conv2d(3, 32, 3, stride=2, bias=False)
        self.bn1 = _bn2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, bias=False)
        self.bn2 = _bn2d(64)

        self.res1 = nn.Conv2d(64, 128, 1, stride=2, bias=False)
        self.res1_bn = _bn2d(128)
        self.b2_sc1 = SepConv(64, 128)
        self.b2_bn1 = _bn2d(128)
        self.b2_sc2 = SepConv(128, 128)
        self.b2_bn2 = _bn2d(128)

        self.res2 = nn.Conv2d(128, 256, 1, stride=2, bias=False)
        self.res2_bn = _bn2d(256)
        self.b3_sc1 = SepConv(128, 256)
        self.b3_bn1 = _bn2d(256)
        self.b3_sc2 = SepConv(256, 256)
        self.b3_bn2 = _bn2d(256)

        self.res3 = nn.Conv2d(256, 728, 1, stride=2, bias=False)
        self.res3_bn = _bn2d(728)
        self.b4_sc1 = SepConv(256, 728)
        self.b4_bn1 = _bn2d(728)
        self.b4_sc2 = SepConv(728, 728)
        self.b4_bn2 = _bn2d(728)

        # --- middle flow: 8 identical blocks ---
        self.mid_sc = nn.ModuleList()
        self.mid_bn = nn.ModuleList()
        for _ in range(8):
            self.mid_sc.append(nn.ModuleList([SepConv(728, 728) for _ in range(3)]))
            self.mid_bn.append(nn.ModuleList([_bn2d(728) for _ in range(3)]))

        # --- exit flow ---
        self.res4 = nn.Conv2d(728, 1024, 1, stride=2, bias=False)
        self.res4_bn = _bn2d(1024)
        self.b13_sc1 = SepConv(728, 728)
        self.b13_bn1 = _bn2d(728)
        self.b13_sc2 = SepConv(728, 1024)
        self.b13_bn2 = _bn2d(1024)
        self.b14_sc1 = SepConv(1024, 1536)
        self.b14_bn1 = _bn2d(1536)
        self.b14_sc2 = SepConv(1536, 2048)
        self.b14_bn2 = _bn2d(2048)

        # --- regression head ---
        self.head_bn1 = _bn1d(2048)
        self.head_drop1 = nn.Dropout(0.4)
        self.head_fc1 = nn.Linear(2048, 256)
        self.head_bn2 = _bn1d(256)
        self.head_drop2 = nn.Dropout(0.3)
        self.head_out = nn.Linear(256, num_targets)

    def features(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        # block2: no activation before the first separable conv
        res = self.res1_bn(self.res1(x))
        x = self.b2_bn1(self.b2_sc1(x))
        x = self.b2_bn2(self.b2_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        for res_c, res_b, sc1, bn1, sc2, bn2 in [
            (self.res2, self.res2_bn, self.b3_sc1, self.b3_bn1, self.b3_sc2, self.b3_bn2),
            (self.res3, self.res3_bn, self.b4_sc1, self.b4_bn1, self.b4_sc2, self.b4_bn2),
        ]:
            res = res_b(res_c(x))
            x = bn1(sc1(F.relu(x)))
            x = bn2(sc2(F.relu(x)))
            x = tf_same_maxpool(x) + res

        for scs, bns in zip(self.mid_sc, self.mid_bn):
            res = x
            for sc, bn in zip(scs, bns):
                x = bn(sc(F.relu(x)))
            x = x + res

        res = self.res4_bn(self.res4(x))
        x = self.b13_bn1(self.b13_sc1(F.relu(x)))
        x = self.b13_bn2(self.b13_sc2(F.relu(x)))
        x = tf_same_maxpool(x) + res

        x = F.relu(self.b14_bn1(self.b14_sc1(x)))
        x = F.relu(self.b14_bn2(self.b14_sc2(x)))
        return x

    def forward_features(self, x):
        """Returns z, the 256-d latent (post-ReLU of Dense(256), pre-final-Dense)."""
        f = self.features(x).mean(dim=(2, 3))   # GlobalAveragePooling2D
        f = self.head_drop1(self.head_bn1(f))
        return F.relu(self.head_fc1(f))

    def forward(self, x):
        z = self.forward_features(x)
        h = self.head_drop2(self.head_bn2(z))
        return self.head_out(h)


def load_encoder_checkpoint(path, num_targets=8, device=DEVICE):
    """Load the ported Xception regressor.

    Strict by design. A key or shape mismatch means the checkpoint was not produced by
    the port above, and a silently half-initialised encoder would poison every latent
    downstream without ever raising.
    """
    ckpt = torch.load(path, map_location='cpu', weights_only=False)
    print(f'Checkpoint top-level type: {type(ckpt)}')

    if isinstance(ckpt, nn.Module):
        print('Checkpoint is a pickled nn.Module; using it directly.')
        return ckpt.to(device)

    if not isinstance(ckpt, dict):
        raise TypeError(f'Unrecognised checkpoint type: {type(ckpt)}')

    state_dict = None
    for key in ('state_dict', 'model', 'model_state_dict'):
        if key in ckpt and isinstance(ckpt[key], dict):
            state_dict = ckpt[key]
            print(f"Found state dict under key '{key}'.")
            break
    if state_dict is None:
        state_dict = ckpt  # bare OrderedDict / state_dict
        print('Treating the top-level dict itself as a bare state_dict.')

    n_out = int(ckpt.get('n_out', num_targets))
    if n_out != num_targets:
        raise ValueError(
            f'Checkpoint was converted with n_out={n_out} but the notebook asks for '
            f'num_targets={num_targets}.'
        )

    enc = XceptionRegressor(num_targets=num_targets)
    enc.load_state_dict(state_dict, strict=True)
    enc = enc.to(device)

    # Provenance written by the conversion notebook -- worth seeing before training on it.
    if 'parity_max_abs_diff' in ckpt:
        print(f"  Keras/Torch parity: max abs diff = {ckpt['parity_max_abs_diff']:.3e} "
              f"(ok={ckpt.get('parity_ok')})")
    if ckpt.get('parity_ok') is False:
        raise RuntimeError(
            'Checkpoint reports parity_ok=False against the Keras model. Re-run '
            'xception-keras-to-torch.ipynb before using it as the cycle encoder.'
        )
    if 'input_size' in ckpt and int(ckpt['input_size']) != 224:
        raise ValueError(
            f"Checkpoint expects input_size={ckpt['input_size']}, but phys_to_encoder_input "
            f'resizes to 224.'
        )
    if 'scalers_match' in ckpt:
        print(f"  DDPM/Xception scalers equivalent: {ckpt['scalers_match']}")

    print(f'  Loaded {len(state_dict)} tensors into XceptionRegressor (strict=True).')
    return enc


encoder = load_encoder_checkpoint(ENCODER_CKPT, num_targets=COND_DIM, device=DEVICE)
encoder.eval()  # BN/Dropout deterministic -- matters even when the weights train (see below)
print(f'Encoder loaded and set to .eval(). Trainable params: '
      f'{sum(p.numel() for p in encoder.parameters() if p.requires_grad):,}')

### 9.1 Preprocessing -- undo the DDPM [-1, 1] normalisation, resize for Xception

```
x_phys = (x_ddpm39 + 1.0) / 2.0 * (mx - mn) + mn
x = F.interpolate(x_phys, size=(224, 224), mode='bilinear', align_corners=False)
x = x.repeat(1, 3, 1, 1)
```
`align_corners=False` matches `tf.image.resize`'s half-pixel convention (SPEC 0.4). No `keras.applications.xception.preprocess_input` and no extra rescaling -- the original training pipeline used the raw pixel range.

In [ ]:
def ddpm_norm_to_phys(x_norm, mn=IMG_MIN, mx=IMG_MAX):
    """Undo the DDPM's [-1, 1] normalisation back to the raw physical pixel range."""
    return (x_norm + 1.0) / 2.0 * (mx - mn) + mn


def phys_to_encoder_input(x_phys):
    """Raw physical pixels (B, 1, 39, 39) -> Xception input (B, 3, 224, 224)."""
    x = F.interpolate(x_phys, size=(224, 224), mode='bilinear', align_corners=False)
    return x.repeat(1, 3, 1, 1)


def encoder_preprocess(x_norm39):
    """DDPM-normalised (B, 1, 39, 39) in [-1, 1] -> Xception input (B, 3, 224, 224)."""
    return phys_to_encoder_input(ddpm_norm_to_phys(x_norm39))


def vendi_score(X, eps=1e-12):
    """Effective number of distinct rows in X, under a cosine kernel.

    Ranges from 1 (every row identical up to scale) to n (mutually orthogonal rows), so
    with EVAL_K=32 draws of one theta the scale is [1, 32]. It is exp of the Shannon
    entropy of the eigenvalues of the normalised Gram matrix: a generator that collapses
    to a single texture for a given theta scores 1 no matter how good its fidelity is.

    This is the guardrail against buying fidelity with diversity. Averaging metrics over K
    draws (which every R^2 here does) HIDES a collapse: if all K draws become identical the
    averaged metrics look excellent while the model has stopped being generative.
    """
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)
    n = Xn.shape[0]
    w = np.linalg.eigvalsh((Xn @ Xn.T) / n)
    w = np.clip(w, 0.0, None)
    w = w[w > eps]
    return float(np.exp(-(w * np.log(w)).sum()))


def shannon_entropy_of(X, eps=1e-12):
    """Shannon entropy (nats) of the same eigenvalue spectrum the Vendi score exponentiates.

    Exactly ``log(vendi_score(X))``: the same quantity on a different scale, reported
    because the entropy in nats is the form used in part of the literature. It carries no
    information the Vendi score does not; the interpretable one is the Vendi score, whose
    units are "effective number of distinct samples".
    """
    return float(np.log(max(vendi_score(X, eps), eps)))


def get_latent(enc, x_norm39):
    """z = the 256-d latent for a batch of DDPM-normalised, ALREADY-CROPPED-TO-39 images.

    Accepts the bare XceptionRegressor, the _EncoderLatent wrapper, or a DataParallel over
    it. Only the wrapper forms can be split across GPUs, because DataParallel dispatches
    ``forward`` and not ``forward_features``.
    """
    x = encoder_preprocess(x_norm39)
    if isinstance(enc, XceptionRegressor):
        return enc.forward_features(x)
    return enc(x)


print('Preprocessing + get_latent defined.')


### 9.2 Mandatory parity check

Published reference R² (from the original Keras training run): KDM 0.9498, J2 0.9146, T 0.8430. If the measured R² is far below these, the preprocessing or the checkpoint key mapping is wrong -- **the notebook stops rather than silently continuing.**

In [ ]:
# Replicate the inverse model's OWN train/val/test split (SPEC 0.4) -- this is a
# DIFFERENT split from the DDPM's 70/15/15 split, and is the split whose train fold
# the target MinMaxScaler was fit on.
_idx_all = np.arange(N)
_idx_trainval_inv, IDX_TEST_INV = train_test_split(_idx_all, test_size=0.15, random_state=42)
IDX_TRAIN_INV, IDX_VAL_INV = train_test_split(_idx_trainval_inv, test_size=0.1765, random_state=42)

INV_SCALER = MinMaxScaler().fit(params[IDX_TRAIN_INV])
print(f'Inverse-model split: train={len(IDX_TRAIN_INV):,}  val={len(IDX_VAL_INV):,}  test={len(IDX_TEST_INV):,}')


REFERENCE_R2 = {'KDM': 0.9498, 'J2': 0.9146, 'T': 0.8430}
R2_WARN_MARGIN = 0.15  # absolute R2 drop tolerated before the notebook halts loudly


@torch.no_grad()
def encoder_parity_r2(enc, idx_subset, batch_size=64, max_samples=1024):
    enc.eval()
    idx_subset = idx_subset[:max_samples]
    preds, targets = [], []
    for i in range(0, len(idx_subset), batch_size):
        idx_b = idx_subset[i:i + batch_size]
        x_phys = torch.from_numpy(imgs[idx_b]).permute(0, 3, 1, 2).float().to(DEVICE)
        y_true = INV_SCALER.transform(params[idx_b])
        x_in = phys_to_encoder_input(x_phys)
        y_pred = enc(x_in).cpu().numpy()
        preds.append(y_pred)
        targets.append(y_true)
    preds = np.concatenate(preds, axis=0)
    targets = np.concatenate(targets, axis=0)
    return r2_score(targets, preds, multioutput='raw_values'), preds, targets


_r2_per_param, _, _ = encoder_parity_r2(encoder, IDX_TEST_INV)
print('Per-parameter R^2 on the inverse-model held-out test split:')
for name, r2 in zip(PARAM_NAMES, _r2_per_param):
    print(f'  {name:8s} R^2 = {r2:.4f}')

# Loud, hard stop if parity is far below the published reference -- do not continue
# training or sampling on a mis-mapped encoder.
# Column indices come from metrics.PARAM_INDEX, the verified order of
# data['params']: T, Jex2, Jex3, Jex4, Kan1, KanS, Hex, KDM.
_param_idx = {'T': PARAM_INDEX['T'], 'J2': PARAM_INDEX['Jex2'], 'KDM': PARAM_INDEX['KDM']}
_bad = []
for ref_name, ref_val in REFERENCE_R2.items():
    if ref_name in _param_idx:
        measured = _r2_per_param[_param_idx[ref_name]]
        if measured < ref_val - R2_WARN_MARGIN:
            _bad.append((ref_name, ref_val, measured))
if _bad:
    msg = ' | '.join(f'{n}: expected~{ref:.4f} got {got:.4f}' for n, ref, got in _bad)
    raise RuntimeError(
        'ENCODER PARITY CHECK FAILED -- preprocessing or checkpoint key mapping is '
        f'almost certainly wrong. {msg}. STOPPING before any further training/sampling.'
    )
print('Parity check passed (within tolerance of published reference values).')


## 10. Freeze everything

Both the DDPM and the encoder are frozen for the rest of this notebook -- only the small `LatentPredictor` head (Step A) is trained from scratch.

In [ ]:
model.requires_grad_(False)
model.eval()
encoder.requires_grad_(False)
encoder.eval()

assert sum(p.requires_grad for p in model.parameters()) == 0, 'DDPM must be frozen'
assert sum(p.requires_grad for p in encoder.parameters()) == 0, 'Encoder must be frozen'
print('DDPM and encoder both frozen.')

MODEL_NAME = 'DDPM+guidance'


#### Dual-GPU execution (Kaggle T4 x2)

This session runs on two T4 GPUs, so both are put to use with `nn.DataParallel`, which splits each batch across the visible devices and gathers the outputs back together; `DistributedDataParallel` is not used because it needs a separate process per GPU launched from outside the notebook, which does not fit this execution model. Splitting the batch is safe for the conditional U-Net because it normalises activations with GroupNorm, which computes its statistics per sample and per group rather than across the batch, so the result is identical regardless of how the batch is partitioned across devices. The encoder is always run in `.eval()` mode, so its BatchNorm layers use their stored running statistics instead of batch statistics, making the encoder split-safe as well. Every checkpoint write, EMA update, and optimiser reference below deliberately targets the unwrapped module rather than the DataParallel wrapper, because wrapping renames every parameter with a `module.` prefix, and this project's checkpoint loader is strict, with no `strict=False` fallback.

In [ ]:
# --- Dual-GPU execution -------------------------------------------------------
# Set to False to force single-GPU even on a multi-GPU machine.
USE_DATA_PARALLEL = True

N_GPU = torch.cuda.device_count()
print(f'Visible CUDA devices: {N_GPU}')
for _i in range(N_GPU):
    print(f'  [{_i}] {torch.cuda.get_device_name(_i)}')


class _EncoderLatent(nn.Module):
    """Exposes ``forward_features`` as ``forward``.

    nn.DataParallel replicates a module and dispatches its ``forward`` only, so a custom
    method such as ``forward_features`` would silently run on a single device. Wrapping it
    keeps the 256-d latent identical while letting the batch be split.
    """

    def __init__(self, enc):
        super().__init__()
        self.enc = enc

    def forward(self, x):
        return self.enc.forward_features(x)


_multi = USE_DATA_PARALLEL and N_GPU > 1

# unet_core / encoder_core are ALWAYS the unwrapped modules. Every state_dict read or
# write, the EMA objects and the optimiser param groups must use these, never the
# DataParallel wrappers, or the saved keys gain a 'module.' prefix and stop loading.
unet_core = model
encoder_core = encoder

_enc_latent = _EncoderLatent(encoder)
if _multi:
    unet_fwd = nn.DataParallel(model)
    encoder_fwd = nn.DataParallel(_enc_latent)
    print(f'DataParallel enabled over {N_GPU} GPUs (batch is split along dim 0).')
else:
    unet_fwd = model
    encoder_fwd = _enc_latent
    print('Single-GPU execution (DataParallel not applied).')

# Guard: the unwrapped module must never carry the DataParallel prefix.
assert not any(k.startswith('module.') for k in unet_core.state_dict()), \
    'unet_core is wrapped; checkpoints would be written with a module. prefix'

## 11. Step A -- train `theta -> z*` (LatentPredictor)

`LatentPredictor`: `Linear(8, 256) -> SiLU -> Linear(256, 512) -> SiLU -> Linear(512, 256)`.
Loss: `(1 - cos(pred, z_target)) + PREDICTOR_MSE_WEIGHT * MSE(pred, z_target)`. This is a lightweight auxiliary network, not a fine-tune of the DDPM or the encoder.

In [ ]:
class LatentPredictor(nn.Module):
    def __init__(self, cond_dim=8, latent_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, 256), nn.SiLU(),
            nn.Linear(256, 512), nn.SiLU(),
            nn.Linear(512, latent_dim),
        )

    def forward(self, theta):
        return self.net(theta)


print('LatentPredictor defined.')


### Precompute `z` for the training split (frozen encoder, no grad, cached to disk)

In [ ]:
@torch.no_grad()
def precompute_latents(enc, imgs_arr, batch_size=64):
    """imgs_arr: raw physical (N, 39, 39, 1) images -> (N, 256) latents."""
    enc.eval()
    zs = []
    for i in range(0, len(imgs_arr), batch_size):
        batch = imgs_arr[i:i + batch_size]
        x = torch.from_numpy(batch).permute(0, 3, 1, 2).float().to(DEVICE)
        x_norm = (x - IMG_MIN) / (IMG_MAX - IMG_MIN + 1e-8) * 2.0 - 1.0
        z = get_latent(enc, x_norm)
        zs.append(z.cpu().numpy())
    return np.concatenate(zs, axis=0)


# Prefer the published cache (dataset weights-latent-predictor): encoding the 118,769
# training images through Xception at 224x224 is the most expensive part of Step A.
LATENT_CACHE_PATH = f'{WORK_DIR}/z_train_cache.npy'
try:
    _published_cache = find_file('z_train_cache.npy')
    if _published_cache and not os.path.exists(LATENT_CACHE_PATH):
        LATENT_CACHE_PATH = _published_cache
        print(f'Using published latent cache: {LATENT_CACHE_PATH}')
except FileNotFoundError:
    print('No published latent cache attached; latents will be recomputed.')
if os.path.exists(LATENT_CACHE_PATH):
    Z_TRAIN = np.load(LATENT_CACHE_PATH)
    print(f'Loaded cached latents: {Z_TRAIN.shape}')
else:
    Z_TRAIN = precompute_latents(encoder_fwd, SPLIT['imgs_tr'])
    np.save(LATENT_CACHE_PATH, Z_TRAIN)
    print(f'Computed and cached latents: {Z_TRAIN.shape} -> {LATENT_CACHE_PATH}')

Z_VAL = precompute_latents(encoder_fwd, SPLIT['imgs_va'])
print(f'Z_VAL: {Z_VAL.shape}')


### Train

In [ ]:
predictor = LatentPredictor(cond_dim=COND_DIM, latent_dim=256).to(DEVICE)
pred_optimizer = torch.optim.Adam(predictor.parameters(), lr=PREDICTOR_LR)

theta_tr = torch.from_numpy(SPLIT['p_tr']).float().to(DEVICE)
z_tr = torch.from_numpy(Z_TRAIN).float().to(DEVICE)
theta_va = torch.from_numpy(SPLIT['p_va']).float().to(DEVICE)
z_va = torch.from_numpy(Z_VAL).float().to(DEVICE)

pred_batch_size = 256
n_train = theta_tr.shape[0]
pred_history = {'train_loss': [], 'val_loss': [], 'val_cosine': []}

# Reuse the published predictor when it is attached. Its held-out cosine is the ceiling
# on what guidance can achieve, so reusing the exact validated weights keeps every sweep
# comparable instead of re-rolling a slightly different predictor each run.
# PREDICTOR_CKPT_IN is the READ-ONLY published checkpoint under /kaggle/input.
# It must not shadow PREDICTOR_CKPT, which is this notebook's writable OUTPUT path.
PREDICTOR_CKPT_IN = None
try:
    PREDICTOR_CKPT_IN = find_file('latent_predictor.pt')
except FileNotFoundError:
    pass

if PREDICTOR_CKPT_IN:
    _pck = torch.load(PREDICTOR_CKPT_IN, map_location=DEVICE, weights_only=False)
    predictor.load_state_dict(_pck['predictor'], strict=True)
    predictor.eval()
    HELD_OUT_COSINE = float(_pck['held_out_cosine'])
    pred_history = _pck.get('history', {'train_loss': [], 'val_loss': [], 'val_cosine': []})
    PREDICTOR_EPOCHS_RUN = 0
    print(f'Loaded published LatentPredictor: held_out_cosine={HELD_OUT_COSINE:.4f} '
          f'(skipping {PREDICTOR_EPOCHS} training epochs)')

for epoch in range(1, (0 if PREDICTOR_CKPT_IN else PREDICTOR_EPOCHS) + 1):
    predictor.train()
    perm = torch.randperm(n_train, device=DEVICE)
    ep_losses = []
    for i in range(0, n_train, pred_batch_size):
        idx_b = perm[i:i + pred_batch_size]
        pred_optimizer.zero_grad(set_to_none=True)
        z_pred = predictor(theta_tr[idx_b])
        cos_sim = F.cosine_similarity(z_pred, z_tr[idx_b], dim=1, eps=1e-8)
        loss = (1.0 - cos_sim).mean() + PREDICTOR_MSE_WEIGHT * F.mse_loss(z_pred, z_tr[idx_b])
        loss.backward()
        pred_optimizer.step()
        ep_losses.append(loss.item())

    predictor.eval()
    with torch.no_grad():
        z_pred_va = predictor(theta_va)
        cos_va = F.cosine_similarity(z_pred_va, z_va, dim=1, eps=1e-8)
        val_loss = (1.0 - cos_va).mean().item() + PREDICTOR_MSE_WEIGHT * F.mse_loss(z_pred_va, z_va).item()
        val_cos_mean = cos_va.mean().item()

    pred_history['train_loss'].append(float(np.mean(ep_losses)))
    pred_history['val_loss'].append(float(val_loss))
    pred_history['val_cosine'].append(float(val_cos_mean))
    if epoch % 5 == 0 or epoch == 1 or epoch == PREDICTOR_EPOCHS:
        print(f"Ep[{epoch:2d}/{PREDICTOR_EPOCHS}] train_loss={pred_history['train_loss'][-1]:.4f} "
              f"val_loss={val_loss:.4f} val_cosine={val_cos_mean:.4f}")

# Keep the checkpoint's value when the predictor was loaded rather than trained; fall back
# to the last validation epoch otherwise. Both paths agree when history is present.
if pred_history.get('val_cosine'):
    HELD_OUT_COSINE = pred_history['val_cosine'][-1]
print(f'\nHeld-out cosine similarity (theta -> z*): {HELD_OUT_COSINE:.4f}')
print('This number bounds how good the guidance can possibly be -- it belongs in the article.')

_sd = predictor.state_dict()
assert not any(k.startswith('module.') for k in _sd), \
    'refusing to save a DataParallel-prefixed state_dict'

torch.save({'predictor': _sd, 'held_out_cosine': HELD_OUT_COSINE,
            'history': pred_history}, PREDICTOR_CKPT)
predictor.eval()
for p in predictor.parameters():
    p.requires_grad_(False)


## 12. Step B -- guided sampling

Classifier-guidance-style: inside the sampling loop, gradient-enabled steps nudge `eps_pred` towards a predicted `x0` with higher cosine similarity to the target latent `z*`. Guidance is applied only while `t < GUIDANCE_T_MAX`, and the gradient is normalised per sample so `GUIDANCE_SCALE` stays scale-free and comparable across runs.

In [ ]:
def guided_sample(unet, enc, cond, predictor_net, sched, guidance_scale,
                   n_steps=SAMPLE_STEPS, img_size=IMG_SIZE, guidance_t_max=GUIDANCE_T_MAX):
    B = cond.shape[0]
    with torch.no_grad():
        z_star = predictor_net(cond)

    x = torch.randn(B, 1, img_size, img_size, device=cond.device)
    timesteps = list(range(0, sched.T, sched.T // n_steps))[::-1]

    for t_val in timesteps:
        t_tensor = torch.full((B,), t_val, device=cond.device, dtype=torch.long)
        sqrt_a = sched.sqrt_alphas_cumprod[t_val]
        sqrt_1a = sched.sqrt_one_minus_alphas_cumprod[t_val]

        if guidance_scale > 0 and t_val < guidance_t_max:
            with torch.enable_grad():
                x_in = x.detach().requires_grad_(True)
                eps_pred = unet(x_in, t_tensor, cond)
                x0_pred = ((x_in - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
                # Crop to 39 BEFORE the encoder ever sees the image.
                # `enc` here is always the single-device encoder (never encoder_fwd):
                # this loop runs one small step at a time, and DataParallel's per-call
                # scatter/gather overhead is not amortised by a single sampling step,
                # so this call deliberately stays on one device. The same applies to
                # `unet` above -- sampling always uses the single-device `model`, never
                # unet_fwd, because a multi-step sampling loop would pay the
                # DataParallel overhead once per step.
                z_pred = get_latent(enc, topleft_crop(x0_pred))
                loss_g = (1.0 - F.cosine_similarity(z_pred, z_star, dim=1, eps=1e-8)).sum()
                grad = torch.autograd.grad(loss_g, x_in)[0]
            # torch.linalg.vector_norm, NOT Tensor.norm: with a dim tuple whose length is
            # not 2, Tensor.norm dispatches to linalg.matrix_norm and raises
            # "dim must be a 2-tuple". This is the per-sample gradient normalisation that
            # keeps GUIDANCE_SCALE scale-free, so it must reduce over C, H and W at once.
            grad_norm = torch.linalg.vector_norm(grad, dim=(1, 2, 3), keepdim=True).clamp_min(1e-8)
            eps_pred = eps_pred.detach() + guidance_scale * sqrt_1a * (grad.detach() / grad_norm)
        else:
            with torch.no_grad():
                eps_pred = unet(x, t_tensor, cond)

        with torch.no_grad():
            x0_pred = ((x - sqrt_1a * eps_pred) / sqrt_a).clamp(-1, 1)
            if t_val > 0:
                t_prev = max(t_val - sched.T // n_steps, 0)
                sqrt_a_prev = sched.sqrt_alphas_cumprod[t_prev]
                sqrt_1a_prev = sched.sqrt_one_minus_alphas_cumprod[t_prev]
                x = sqrt_a_prev * x0_pred + sqrt_1a_prev * eps_pred
            else:
                x = x0_pred

    return x.detach(), z_star.detach()


# Sanity check: GUIDANCE_SCALE=0 must reproduce plain (unguided) fast_sample bit-for-bit
# given the same random seed and starting noise.
torch.manual_seed(SEED)
_cond_check = torch.from_numpy(SPLIT['p_va'][:4]).float().to(DEVICE)
torch.manual_seed(123)
_x_guided0, _ = guided_sample(model, encoder, _cond_check, predictor, scheduler, guidance_scale=0, n_steps=100)
torch.manual_seed(123)
_x_plain = fast_sample(model, _cond_check, scheduler, n_steps=100, img_size=IMG_SIZE)
_max_diff = (_x_guided0 - _x_plain).abs().max().item()
print(f'Max abs diff, guidance_scale=0 vs plain fast_sample: {_max_diff:.3e} (expected ~0)')
assert _max_diff < 1e-4, 'guidance_scale=0 must reproduce the plain DDPM'
del _cond_check, _x_guided0, _x_plain


## 13. Step C -- guidance-scale sweep

For each scale in `GUIDANCE_SCALES`, report physical-metric R^2 (generated vs. original), SSIM, latent cosine (generated vs. `z*`), and cycle R^2/MAE per parameter. Expect texture artefacts with rising latent cosine at high scale -- that failure is reported, not hidden.

In [ ]:
# IDX_TEST_INV / INV_SCALER were already built in the encoder parity-check cell above
# (SPEC 0.4 inverse-model split). Only a small predict helper is needed here to turn
# encoder+head predictions back into physical parameter units for the sweep below.
@torch.no_grad()
def encoder_predict_theta_sweep(x_norm39_np, batch_size=64):
    preds = []
    for i in range(0, len(x_norm39_np), batch_size):
        xb = torch.from_numpy(x_norm39_np[i:i + batch_size]).unsqueeze(1).float().to(DEVICE)
        y = encoder(encoder_preprocess(xb)).cpu().numpy()
        preds.append(y)
    return np.concatenate(preds, axis=0)


print(f'Reusing inverse-model split: train={len(IDX_TRAIN_INV):,}  test={len(IDX_TEST_INV):,}')


## Comparison of the four architectures

All four are evaluated in one cell, on the same stratified thetas, with the same
seed and the same code path, so the rows are comparable by construction.


In [ ]:
_, _, ds_te, _, _, dl_te = make_dataloaders(SPLIT, batch_size=BEST_HPARAMS['batch_size'])
n_test_total = len(ds_te)
n_test_eval = max(1, int(n_test_total * TEST_FRACTION))
print(f'Sweep evaluated on {n_test_eval:,} / {n_test_total:,} test samples (TEST_FRACTION={TEST_FRACTION}).')

orig_39_all, cond_all = [], []
n_seen = 0
for x0, cond in dl_te:
    if n_seen >= n_test_eval:
        break
    orig_39_all.append(topleft_crop(x0).numpy()[:, 0])
    cond_all.append(cond.numpy())
    n_seen += x0.shape[0]
orig_39 = np.concatenate(orig_39_all, axis=0)[:n_test_eval]
cond_te_np = np.concatenate(cond_all, axis=0)[:n_test_eval]
cond_te = torch.from_numpy(cond_te_np).float().to(DEVICE)
assert orig_39.shape[-2:] == (39, 39)
phys_orig = physical_metrics_batch(orig_39)
theta_true = SPLIT['scaler'].inverse_transform(cond_te_np)

sweep_results = {}
# Images per generation call. EVAL_K samples of one theta must fit together, and the
# guided branch backpropagates through Xception at 224x224, so this cannot be large.
sweep_batch = 64
theta_chunk = max(1, sweep_batch // EVAL_K)

# Snapshot of the FULL test arrays before the stratified subset replaces them. The
# real-data diversity reference below needs neighbours drawn from the whole test split,
# not from the 1,000 thetas that happened to be selected.
_ORIG_FULL = orig_39
_PHYS_FULL = phys_orig

# Proportional (stratified) selection by magnetic structure, replicating the reference:
# take round(N_THETA_EVAL * phase_fraction) from each phase so the evaluated subset keeps
# the test split's phase composition. Taking the first N instead leaves the composition to
# chance, and peak_wave_vector is ill-defined on near-uniform configurations, so its R^2
# is the metric most sensitive to how the subset was drawn.
if labels is not None:
    _te_lab = np.array([get_structure_label(c) for c in labels[SPLIT['idx_te']]])[:cond_te.shape[0]]
    _rng_bal = np.random.RandomState(SEED)
    _sel = []
    print('Stratified selection by structure:')
    for _ph in sorted(set(_te_lab.tolist())):
        _idx_in = np.where(_te_lab == _ph)[0]
        _n_take = min(max(1, int(round(N_THETA_EVAL * len(_idx_in) / len(_te_lab)))), len(_idx_in))
        _sel.append(_rng_bal.choice(_idx_in, size=_n_take, replace=False))
        print(f'  {_ph:24s} {_n_take:5,}')
    _eval_idx = np.concatenate(_sel); _rng_bal.shuffle(_eval_idx)
    print(f'  TOTAL: {len(_eval_idx):,}')
else:
    print('No labels array; falling back to the first N_THETA_EVAL test samples.')
    _eval_idx = np.arange(min(N_THETA_EVAL, cond_te.shape[0]))

n_theta = len(_eval_idx)
cond_eval = cond_te[_eval_idx]
orig_39 = orig_39[_eval_idx]
phys_orig = {k: v[_eval_idx] for k, v in phys_orig.items()}
theta_true = theta_true[_eval_idx]
EVAL_IDX = _eval_idx   # reused by the per-phase breakdown so the labels line up

# --- Upper reference: diversity of REAL images with NEARBY theta -----------------------
# A high Vendi score is not automatically good: the DDPM should reproduce the true
# conditional variability, not maximise diversity. This quantity was built as a target,
# but a simulation showed it does NOT work as one. The EVAL_K nearest neighbours have
# DIFFERENT theta, so their textures differ because of the conditioning, not because of
# stochasticity: with a nearly deterministic ground truth (sigma=0.02) this reference
# still scores ~21/32 while a generator that reproduces that variability correctly scores
# ~1. The ratio is therefore NOT expected to approach 1, and must not be read as
# "matches reality".
# What it IS: a loose UPPER bound. A generator whose Vendi exceeds it is clearly too
# diverse. The trustworthy signal remains the TREND of the Vendi score across
# GUIDANCE_SCALE -- if it falls towards 1 as the scale rises, fidelity was bought with
# the model's stochasticity. A tight reference would need several real realisations of
# the SAME theta, which this dataset does not obviously provide.
from sklearn.neighbors import NearestNeighbors

_cond_pool = cond_te_np   # scaled thetas of the whole test split
_nn = NearestNeighbors(n_neighbors=EVAL_K).fit(_cond_pool)
_, _nbr = _nn.kneighbors(_cond_pool[EVAL_IDX])

REAL_VENDI_PIX = float(np.nanmean([
    vendi_score(_ORIG_FULL[_nbr[j]].reshape(EVAL_K, -1)
                - _ORIG_FULL[_nbr[j]].reshape(EVAL_K, -1).mean(axis=1, keepdims=True))
    for j in range(len(EVAL_IDX))]))
REAL_WITHIN_STD = {name: float(np.nanmean(np.nanstd(_PHYS_FULL[name][_nbr], axis=1)))
                   for name in PHYSICAL_METRIC_NAMES}
print(f'Upper reference from {EVAL_K} nearest-theta REAL neighbours (NOT a target):')
print(f'  vendi_pixel(neighbours) = {REAL_VENDI_PIX:.2f} / {EVAL_K}')
print(f'  within-theta std(real) = { {k: round(v, 4) for k, v in REAL_WITHIN_STD.items()} }')
print(f'Sweep: {n_theta} thetas x EVAL_K={EVAL_K} samples = {n_theta * EVAL_K:,} generations '
      f'per scale, {SAMPLE_STEPS} steps, theta_chunk={theta_chunk}.')


# =============================================================================
#  Four architectures, ONE protocol
# =============================================================================
# Every variant is scored inside THIS cell, on the same stratified thetas, with the same
# seed, the same EVAL_K draws and the same code path. Four separate notebooks cannot give
# that guarantee: a single differing constant silently makes the rows incomparable, which
# is exactly what happened when one run averaged K draws and another scored a single one.
GUIDANCE_SCALE_EVAL = 8.0

def _try(name):
    try:
        return find_file(name)
    except FileNotFoundError:
        return None

# ONE model per notebook. The 4-model sweep does not fit a single session: the
# measured run at this protocol was cancelled at the 12 h cap after only two
# scales (08:05 -> 20:17), so the guided variant alone costs ~10.5 h. Splitting
# keeps the protocol IDENTICAL -- same SEED, same stratified thetas, same EVAL_K,
# same code -- while letting each variant run in its own session, on its own
# account, in parallel.
MODEL_SPECS = [
    ('DDPM base', 'ddpm_spines_final_39crop.pt', 0.0),
]

comparison = {}
sweep_batch = 64
theta_chunk = max(1, sweep_batch // EVAL_K)

for MODEL_LABEL, ckpt_name, gscale in MODEL_SPECS:
    path = _try(ckpt_name)
    if path is None:
        print(f'SKIP {MODEL_LABEL}: {ckpt_name} not attached.')
        continue
    print(f'\n=== {MODEL_LABEL}  (guidance={gscale})  <- {ckpt_name} ===', flush=True)

    _ck = torch.load(path, map_location=DEVICE, weights_only=False)
    _st = _ck['model'] if isinstance(_ck, dict) and 'model' in _ck else _ck
    model.load_state_dict(_st, strict=True)
    model.eval()

    torch.manual_seed(SEED)   # same noise draws for every variant
    met_k = {n: np.full((n_theta, EVAL_K), np.nan) for n in PHYSICAL_METRIC_NAMES}
    theta_k = np.full((n_theta, EVAL_K, COND_DIM), np.nan)
    ssim_k = np.full((n_theta, EVAL_K), np.nan)
    mse_k = np.full((n_theta, EVAL_K), np.nan)
    vendi_lat = np.full(n_theta, np.nan)
    vendi_pix = np.full(n_theta, np.nan)
    first_draw = []   # one draw per theta, for the figures

    for i in range(0, n_theta, theta_chunk):
        cond_b = cond_eval[i:i + theta_chunk]
        c = cond_b.shape[0]
        cond_rep = cond_b.repeat_interleave(EVAL_K, dim=0)
        x_gen, _zs = guided_sample(model, encoder, cond_rep, predictor, scheduler,
                                   guidance_scale=gscale, n_steps=SAMPLE_STEPS)
        gen_b = topleft_crop(x_gen)
        gen_np = gen_b.cpu().numpy()[:, 0]

        m_b = physical_metrics_batch(gen_np)
        for n in PHYSICAL_METRIC_NAMES:
            met_k[n][i:i + c] = m_b[n].reshape(c, EVAL_K)

        with torch.no_grad():
            _z = get_latent(encoder_fwd, gen_b)
            th_b = encoder_core.head_out(encoder_core.head_bn2(_z)).cpu().numpy()
        theta_k[i:i + c] = th_b.reshape(c, EVAL_K, th_b.shape[-1])

        _zc = _z.cpu().numpy().reshape(c, EVAL_K, -1)
        _pc = gen_np.reshape(c, EVAL_K, -1)
        for j in range(c):
            vendi_lat[i + j] = vendi_score(_zc[j])
            vendi_pix[i + j] = vendi_score(_pc[j] - _pc[j].mean(axis=1, keepdims=True))
            ref = orig_39[i + j]
            blk = gen_np[j * EVAL_K:(j + 1) * EVAL_K]
            ssim_k[i + j] = [masked_ssim(ref, g) for g in blk]
            mse_k[i + j] = [masked_mse(ref, g) for g in blk]
        first_draw.append(gen_np.reshape(c, EVAL_K, 39, 39)[:, 0])

    phys_gen = {n: np.nanmean(v, axis=1) for n, v in met_k.items()}
    phys_r2 = {}
    for n in PHYSICAL_METRIC_NAMES:
        o, g = phys_orig[n], phys_gen[n]
        ok = np.isfinite(o) & np.isfinite(g)
        phys_r2[n] = float(r2_score(o[ok], g[ok])) if ok.sum() > 1 else float('nan')

    theta_hat = INV_SCALER.inverse_transform(np.nanmean(theta_k, axis=1))
    cyc_r2 = r2_score(theta_true, theta_hat, multioutput='raw_values')
    cyc_mae = mean_absolute_error(theta_true, theta_hat, multioutput='raw_values')

    comparison[MODEL_LABEL] = {
        'checkpoint': ckpt_name, 'guidance_scale': gscale,
        'physical_r2': phys_r2,
        'ssim_mean': float(np.nanmean(ssim_k)),
        'masked_mse_mean': float(np.nanmean(mse_k)),
        'cycle_r2': {n: float(v) for n, v in zip(PARAM_NAMES, cyc_r2)},
        'cycle_mae': {n: float(v) for n, v in zip(PARAM_NAMES, cyc_mae)},
        'cycle_r2_mean': float(np.mean(cyc_r2)),
        'vendi_latent_mean': float(np.nanmean(vendi_lat)),
        'vendi_pixel_mean': float(np.nanmean(vendi_pix)),
        'vendi_max_possible': EVAL_K,
        'vendi_pixel_neighbour_upper_bound': REAL_VENDI_PIX,
        'within_theta_std': {n: float(np.nanmean(np.nanstd(v, axis=1))) for n, v in met_k.items()},
        'within_theta_std_real': REAL_WITHIN_STD,
        'eval_k': EVAL_K, 'n_theta': int(n_theta), 'sample_steps': int(SAMPLE_STEPS),
    }
    # The downstream figures show ONE model; use the baseline so the s_z panel and the
    # parity plot always depict the reference generator rather than whichever variant
    # happened to run last.
    # One model per notebook, so the figures always depict THIS variant. The earlier
    # version only assigned gen_39 for the baseline label, which left every other
    # notebook with an undefined name at the s_z figure.
    gen_39 = np.concatenate(first_draw, axis=0)

    r = comparison[MODEL_LABEL]
    print(f"  physical R2 = { {k: round(v,4) for k,v in phys_r2.items()} }")
    print(f"  cycle R2 = {r['cycle_r2_mean']:.4f}   SSIM = {r['ssim_mean']:.4f}")
    print(f"  vendi pixel = {r['vendi_pixel_mean']:.2f}/{EVAL_K} "
          f"(real neighbours {REAL_VENDI_PIX:.2f})   vendi latent = {r['vendi_latent_mean']:.2f}")

    with open(f'{WORK_DIR}/comparison_ddpm-base.json', 'w') as f:
        json.dump(comparison, f, indent=2)


In [ ]:
# --- The comparison table -----------------------------------------------------------
_cols = ['magnetization', 'spin_correlation', 'peak_wave_vector']
print(f"{'model':24s} {'magnet':>8} {'C_nn':>8} {'q_peak':>8} {'SSIM':>8} "
      f"{'cycleR2':>8} {'V_pix':>7} {'V_lat':>7}")
print('-' * 86)
for _name, _r in comparison.items():
    _p = _r['physical_r2']
    print(f"{_name:24s} {_p[_cols[0]]:>8.4f} {_p[_cols[1]]:>8.4f} {_p[_cols[2]]:>8.4f} "
          f"{_r['ssim_mean']:>8.4f} {_r['cycle_r2_mean']:>8.4f} "
          f"{_r['vendi_pixel_mean']:>7.2f} {_r['vendi_latent_mean']:>7.2f}")
print('-' * 86)
print(f"{'real neighbours (ref)':24s} {'':>8} {'':>8} {'':>8} {'':>8} {'':>8} "
      f"{REAL_VENDI_PIX:>7.2f}")
print(f"\nEVAL_K={EVAL_K}  N={n_theta} stratified  steps={SAMPLE_STEPS}  seed={SEED}")
print('q_peak here is the MASKED, mean-subtracted physical variant from metrics.py.')
print('It is NOT the quantity physical_metrics_3ddpm_comparison reports -- that notebook')
print('calls structure_factor with its defaults (raw image, no disk mask, no mean removed)')
print('and returns the raw radial bin, so the two q_peak columns must never be compared.')

with open(f'{WORK_DIR}/comparison_ddpm-base.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print(f'\nSaved {WORK_DIR}/comparison_ddpm-base.json')

# --- Figure: the three axes side by side --------------------------------------------
_names = list(comparison)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
_x = np.arange(len(_names)); _w = 0.26
for _k, _m in enumerate(_cols):
    axes[0].bar(_x + (_k - 1) * _w, [comparison[n]['physical_r2'][_m] for n in _names], _w, label=_m)
axes[0].set_xticks(_x); axes[0].set_xticklabels(_names, rotation=20, ha='right', fontsize=8)
axes[0].set_ylabel('R^2'); axes[0].set_title('Physical observables'); axes[0].legend(fontsize=7)

axes[1].bar(_x, [comparison[n]['cycle_r2_mean'] for n in _names], color='tab:blue')
axes[1].set_xticks(_x); axes[1].set_xticklabels(_names, rotation=20, ha='right', fontsize=8)
axes[1].set_ylabel('mean cycle R^2'); axes[1].set_title('Parameter recovery')

axes[2].bar(_x - 0.2, [comparison[n]['vendi_pixel_mean'] for n in _names], 0.4, label='pixel')
axes[2].bar(_x + 0.2, [comparison[n]['vendi_latent_mean'] for n in _names], 0.4, label='latent')
axes[2].axhline(REAL_VENDI_PIX, ls='--', c='k', lw=1, label='real neighbours')
axes[2].axhline(1.0, ls=':', c='r', lw=1, label='collapse')
axes[2].set_xticks(_x); axes[2].set_xticklabels(_names, rotation=20, ha='right', fontsize=8)
axes[2].set_ylabel(f'Vendi  [1, {EVAL_K}]'); axes[2].set_title('Diversity'); axes[2].legend(fontsize=7)

fig.suptitle('Four architectures, one protocol')
fig.tight_layout()
save_figure(fig, f'{WORK_DIR}/comparison_ddpm-base')
plt.show()


### Sweep plots and saved results

## Final evaluation (shared protocol, `metrics.py` only)

Canonical Section-3 evaluation block, run at `DEFAULT_EVAL_SCALE` (the sweep above already reports physical R^2 / SSIM / latent cosine / cycle R^2 / MAE for every scale). Physical metrics are computed strictly on the 39x39 crop.

#### Physical-metric agreement on the test split


In [ ]:
_phys_metrics_table_rows = []
phys_r2_table = {}
for _name in PHYSICAL_METRIC_NAMES:
    o, g = phys_orig[_name], phys_gen[_name]
    valid = np.isfinite(o) & np.isfinite(g)
    n_valid = int(valid.sum())
    o_v, g_v = o[valid], g[valid]
    if n_valid > 1:
        r2 = float(r2_score(o_v, g_v))
        pearson_r = float(np.corrcoef(o_v, g_v)[0, 1])
        mae = float(mean_absolute_error(o_v, g_v))
        rmse = float(np.sqrt(mean_squared_error(o_v, g_v)))
    else:
        r2 = pearson_r = mae = rmse = float('nan')
    if n_valid > 0:
        bias = float(g_v.mean() - o_v.mean())
        o_mean, o_std = float(o_v.mean()), float(o_v.std())
        g_mean, g_std = float(g_v.mean()), float(g_v.std())
    else:
        bias = o_mean = o_std = g_mean = g_std = float('nan')

    phys_r2_table[_name] = {
        'r2': r2, 'pearson_r': pearson_r, 'mae': mae, 'rmse': rmse, 'bias': bias,
        'orig_mean': o_mean, 'orig_std': o_std, 'gen_mean': g_mean, 'gen_std': g_std,
        'n': n_valid,
    }
    _phys_metrics_table_rows.append(
        (_name, r2, pearson_r, mae, rmse, bias, o_mean, o_std, g_mean, g_std, n_valid))

_header = (f'{"metric":22s} {"R2":>8s} {"pearson_r":>10s} {"MAE":>8s} {"RMSE":>8s} '
           f'{"bias":>8s} {"orig mean+-std":>18s} {"gen mean+-std":>18s} {"n":>6s}')
print(_header)
print('-' * len(_header))
for (_name, r2, pearson_r, mae, rmse, bias, o_mean, o_std, g_mean, g_std, n_valid) in _phys_metrics_table_rows:
    print(f'{_name:22s} {r2:8.4f} {pearson_r:10.4f} {mae:8.4f} {rmse:8.4f} '
          f'{bias:+8.4f} {o_mean:+7.3f}+-{o_std:<6.3f} {g_mean:+7.3f}+-{g_std:<6.3f} {n_valid:6d}')


#### s_z projections: original vs generated

These are the central-layer $s_z$ projections of the nanodot (z = floor(L/2)), cropped to the 39x39 physical grid; each column is annotated with that sample's physical-metric values.


In [ ]:
N_SHOW = 8
_idx_show = np.linspace(0, len(orig_39) - 1, N_SHOW).astype(int)
# 'jet' with the scale pinned to [-1, 1] on EVERY row, difference row included.
# Fixing the limits is what makes the panels comparable at a glance: s_z = 0
# always lands on the same green, so the background of every image reads the
# same and a colour can be compared across models and across theta.
fig, axes = plt.subplots(3, N_SHOW, figsize=(2.1 * N_SHOW, 7.0))
for _col, _i in enumerate(_idx_show):
    _o, _g = orig_39[_i], gen_39[_i]
    _d = np.abs(_o - _g)
    for _row, (_img, _cmap, _vmin, _vmax) in enumerate(
            [(_o, 'jet', -1.0, 1.0), (_g, 'jet', -1.0, 1.0), (_d, 'jet', -1.0, 1.0)]):
        _ax = axes[_row, _col]
        _im = _ax.imshow(_img, cmap=_cmap, vmin=_vmin, vmax=_vmax, interpolation='nearest')
        _ax.set_xticks([]); _ax.set_yticks([])
    axes[0, _col].set_title(
        '\n'.join(f'{_n}={phys_orig[_n][_i]:+.3f}' for _n in PHYSICAL_METRIC_NAMES), fontsize=6)
    axes[2, _col].set_xlabel(
        '\n'.join(f'{_n}={phys_gen[_n][_i]:+.3f}' for _n in PHYSICAL_METRIC_NAMES), fontsize=6)
axes[0, 0].set_ylabel('original $s_z$', fontsize=9)
axes[1, 0].set_ylabel('generated $s_z$', fontsize=9)
axes[2, 0].set_ylabel('|difference|', fontsize=9)
fig.suptitle(f'{MODEL_NAME}: central-layer $s_z$ projections '
             f'(top titles = original metrics, bottom labels = generated metrics)')
fig.tight_layout()
save_figure(fig, f'{WORK_DIR}/sz_projections')
plt.show()
